# English↔Chinese: fine-tune and evaluate the whole system (one-time run)

This notebook does for English↔Chinese what the paper's project does for the other languages, on Kaggle's free GPUs:

0. **Pre-flight (about 20 min):** every later step is run once at a tiny scale with the real models, so that a problem shows up in the first half hour and not after an hour of training
1. **Fine-tune** TowerInstruct-7B on BMELD-train with the conversation as context (LoRA adapter, the light version of the paper's fine-tuning)
2. **Evaluate** like the paper: base model greedy, then the fine-tuned model with greedy decoding, 6 sampled candidates with COMET MBR
   (plain and context-aware), contrastive decoding and P-CXMI
3. **Consolidate** everything into one results table (`results_zh/bmeld/test.en-zh.csv`, like the paper's `paper_results/*.csv`)

Run it **once**. The adapter (about 0.2 GB) is kept as the notebook's output; afterwards `translate_zh_kaggle.ipynb` answers your own
conversations with one cell, without training or evaluating again. The judge (Gemini) runs in its own CPU notebook, `judge_zh_kaggle.ipynb`.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**. Best started with **Save Version → Save & Run All (Commit)**: it runs in the background
(you can close the browser) and keeps the outputs. If the pre-flight fails, the run stops within about 30 minutes and the log shows the failing step.
Expect about 3 hours of GPU time with the defaults (the notebook prints the real times). During the run about 30 GB of temporary disk is used
(model copies); only the adapter and the small result files stay as output. **After the training, a failing evaluation step does not stop the notebook:**
the adapter is saved first, failed steps are listed at the end.
Optional: attach the secret `GEMINI_API_KEY` (Add-ons → Secrets) and the pre-flight also tests the judge.

In [ ]:
# 1. GPUs (should list two Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code + the en<->zh data only (the full repo is ~4.4 GB; sparse checkout keeps it small)
import os, subprocess
REPO = "namannamu18/context-aware-translation"
BRANCH = "master"
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass                    # public repo: no token needed
url = f"https://{token+'@' if token else ''}github.com/{REPO}.git"
os.chdir("/kaggle/working")
if not os.path.exists("cat"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "-b", BRANCH, url, "cat"], check=True)
os.chdir("/kaggle/working/cat")
dirs = ["scripts", "configs/zh", "paper_results_zh"] + [
    f"raw_data/mt/{ds}.{x}" for ds in ["bmeld_test", "bmeld_dev", "bmeld_train", "synthetic_chat_test"] for x in ["zh", "en-zh", "zh-en"]]
subprocess.run(["git", "sparse-checkout", "set"] + dirs, check=True)
!ls scripts | head -50

In [ ]:
# 3. Python 3.11 environment with the pinned libraries. Kaggle's default Python is newer (3.13) than vLLM 0.6.4 / torch 2.5.1 / numpy<2 support,
#    so everything below runs in its own Python 3.11 environment (made with uv, a few minutes). The notebook's own Python is not touched.
import os, subprocess
VENV = "/kaggle/temp/venv311"
PKGS = ('"torch==2.5.1" "vllm==0.6.4.post1" "transformers==4.46.3" "unbabel-comet==2.2.7" "sacrebleu>=2.4" "numpy<2" "protobuf<5" pyyaml '
        '"peft>=0.13,<0.16" "bitsandbytes>=0.44,<0.46" "accelerate>=1.0,<1.9" "huggingface_hub<1.0" "setuptools>=65,<80" wheel "pyarrow<18" "pandas<2.3" openai')
def run(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode:
        raise RuntimeError(f"failed (exit code {r.returncode}): {cmd[:120]}")
os.makedirs("/kaggle/temp", exist_ok=True)
if not os.path.exists(f"{VENV}/bin/python"):
    run("pip install -q uv")
    run(f"uv venv --python 3.11 {VENV}")
run(f"uv pip install -q --python {VENV}/bin/python {PKGS}")
CHECK = ("import sys, torch, vllm, transformers, peft, bitsandbytes, comet, accelerate; "
         "print('python', sys.version.split()[0], '| torch', torch.__version__, '| vllm', vllm.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)")
subprocess.run([f"{VENV}/bin/python", "-c", CHECK], check=True)
print("environment ready:", VENV)
# If this cell fails: Run -> Restart session, then run cells 1-3 again.

In [ ]:
import os, re, subprocess, time
T0 = time.time()
FAILED = []
BAR_PCT = re.compile(r"\d+%\|[^\]]*\]")          # "37%|####   | 3/8 [00:11<00:20,  4.1s/it]"
BAR_CNT = re.compile(r"\d+it \[[^\]]*\]")           # "12it [00:03,  3.9it/s]"

def sh(cmd, env=None, keep=None, label=None, soft=False, stdin=None):
    """Run a shell command and stream its output without progress bars (keep = regex: show only matching lines).
    On failure the last lines of the full log are shown and the notebook stops (soft=True: the failure is recorded in FAILED and the notebook goes on)."""
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdin=subprocess.PIPE if stdin is not None else None,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
    if stdin is not None:
        p.stdin.write(stdin.encode("utf-8"))
        p.stdin.close()
    log, buf = [], b""
    def handle(raw):
        line = BAR_CNT.sub("", BAR_PCT.sub("", raw.decode("utf-8", "replace"))).strip()   # cut out progress bars, keep what else is on the line
        if not line:
            return
        log.append(line)
        if keep is None or re.search(keep, line):
            print(line, flush=True)
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        *parts, buf = re.split(rb"[\r\n]", buf + chunk)
        for part in parts:
            handle(part)
    handle(buf)
    rc = p.wait()
    if rc:
        print("\n".join(log[-60:]))
        if soft:
            FAILED.append(label or cmd[:60])
            print(f"!!! STEP FAILED (exit code {rc}), continuing: {label or cmd[:100]}", flush=True)
            return False
        raise RuntimeError(f"command failed (exit code {rc}): {cmd[:150]}")
    print(f"[{(time.time() - T0) / 60:.1f} min since start] {label or 'done'}", flush=True)
    return True

# 4. Settings
os.chdir("/kaggle/working/cat")
BASE_MODEL   = "Unbabel/TowerInstruct-7B-v0.2"
FT_NAME      = "TowerInstruct-7B-w-chat-zh"           # name of the fine-tuned system (like TowerInstruct-7B-w-chat in the paper)
ADAPTER_DIR  = "/kaggle/working/zh_lora"              # kept as notebook output (small)
MERGED_DIR   = "/kaggle/temp/tower_zh_merged"         # temporary copy with the adapter merged in (~13 GB, not kept)
RUN_ROOT     = "/kaggle/working/run_zh"               # all outputs of the evaluation (same folder layout as the paper's repo)
SMOKE_LORA   = "/kaggle/temp/smoke_lora"              # adapter of the 4-step pre-flight training (temporary)
PREFLIGHT_ROOT = "/kaggle/temp/preflight"
assert "chat" not in MERGED_DIR.lower() and "chat" not in BASE_MODEL.lower()   # the P-CXMI scripts choose Tower token ids from the model PATH

TRAIN_MINUTES = 60    # fine-tuning budget: stops after one epoch or when this time is used up, whichever comes first
TRAIN_GPUS    = 2     # 1 = use only one T4 (set this if the 2-GPU launch hangs)
EVAL_DOCS     = 12    # BMELD-test conversations (about 7-8 messages each) used for the evaluation
N_CANDIDATES  = 6     # sampled candidates per message (paper: 100)
PREFLIGHT     = True  # keep True for the first run; it costs about 20 minutes and saves you from finding problems after the training
TRANSLATE_EXTRA = ""  # extra command-line options for the translator check

os.environ.update({
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],   # every command below (python, torchrun, the pipeline scripts) uses the Python 3.11 environment
    "VIRTUAL_ENV": VENV,
    "PYTHON": f"{VENV}/bin/python",
    "BACKEND": "vllm",
    "COMET_MODEL": "Unbabel/wmt22-comet-da",
    "UTILITY": "comet",
    "SEP_TOKENS": "tower",                      # Tower token ids for P-CXMI gating (as in the paper)
    # T4: no bfloat16 -> float16; split the 7B model over the two T4s; leave room for P-CXMI's prompt log-probs
    "VLLM_ENGINE_ARGS": '{"dtype": "half", "tensor_parallel_size": 2, "max_model_len": 4096, "gpu_memory_utilization": 0.75, "max_num_seqs": 8, "max_num_batched_tokens": 4096}',
    "CD_EXTRA_ARGS": "--torch_dtype float16 --device_map auto",
    "TOKENIZERS_PARALLELISM": "false",
    "LOOP_RETRY_PENALTY": "1.1",       # only outputs stuck in a repetition loop are translated again (paper setup otherwise)
    "USE_TF": "0", "TRANSFORMERS_NO_TF": "1",   # Kaggle's preinstalled TensorFlow breaks with protobuf<5 (needed by COMET)
    "NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1",   # safer multi-GPU communication on Kaggle's T4s
})
try:   # optional: with the Gemini key (Kaggle Secrets) the pre-flight also tests the judge
    from kaggle_secrets import UserSecretsClient
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    pass
LAUNCH = f"{VENV}/bin/torchrun --nproc_per_node {TRAIN_GPUS}" if TRAIN_GPUS > 1 else f"{VENV}/bin/python"
print("settings ok | judge key:", "found" if os.environ.get("GEMINI_API_KEY") else "not attached (the pre-flight skips the judge)")

In [ ]:
# 5. Download TowerInstruct-7B once (about 14 GB; training and vLLM then read it from the cache)
# (run as a separate process: a download inside the notebook kernel makes the kernel unresponsive and "Save & Run All" aborts with "Timeout waiting for IOPub output")
sh(f"{VENV}/bin/huggingface-cli download {BASE_MODEL} --exclude '*.bin' '*.pt' '*.h5' '*.msgpack'", label="model downloaded")
print("base model cached")

In [ ]:
# 6. PRE-FLIGHT (about 20 minutes): every step that comes later is run once at a tiny scale with the REAL models.
#    Any failure stops the notebook here (the later steps are not run), so you lose minutes, not hours.
if PREFLIGHT:
    # 6a. training: 4 steps (tests the 2-GPU launch, 4-bit loading, the dev-loss evaluation and saving); the adapter is only used by 6d
    sh(f"{LAUNCH} scripts/finetune_lora.py --base_model {BASE_MODEL} --out_dir {SMOKE_LORA} --max_examples 64 --max_steps 4 --eval_examples 16 --save_steps 1000",
       keep=r"train examples|trainable|dev loss|FINETUNE_DONE|Error", label="6a training works")
    # 6b. the paper's pipeline on 2 conversations with the base model and the fine-tuned model's prompts: vLLM, candidates, plain and context-aware
    #     COMET MBR, contrastive decoding, P-CXMI, report
    sh("bash scripts/run_zh_pipeline.sh", keep=r"^=+ |^###|Error|Traceback|FAILED",
       env=dict(ROOT=PREFLIGHT_ROOT, DATA_NAME="bmeld", SPLIT="test", MAX_DOCS="2", MAX_TOKENS="64", MODEL=BASE_MODEL, MODEL_NAME="preflight",
                PROMPT_SUFFIX="_empty_sys", N_CANDIDATES="2", CONTEXT_SIZES="2", CD_VARIANTS="c1_nc1"), label="6b pipeline works")
    # 6c. the consistency checks (with the real tokenizer: P-CXMI gating, prompt alignment, MBR selections) and the results table
    sh(f"python scripts/check_zh_pipeline_outputs.py --root_dir {PREFLIGHT_ROOT} --data_name bmeld --split test --model {BASE_MODEL} --model_name preflight "
       f"--gen_backend vllm --n_candidates 2 --suffix _empty_sys", keep=r"FAIL|checks passed", label="6c checks pass")
    sh(f"python scripts/consolidate_zh_results.py --root_dir {PREFLIGHT_ROOT} --ft_model_name preflight --ft_suffix _empty_sys --gen_backend vllm --comet_model {os.environ['COMET_MODEL']}",
       keep=r"wrote|skipping", label="6c results table works")
    # 6d. the translator of translate_zh_kaggle.ipynb: base model + adapter + COMET (+ judge if the key is attached), reading the adapter folder
    sh(f"{VENV}/bin/python scripts/translate_chat_pipeline.py --adapter {SMOKE_LORA} --n_candidates 3 {TRANSLATE_EXTRA}" + (" --judge gemini" if os.environ.get("GEMINI_API_KEY") else ""),
       stdin="en: Hi, I bought a coat last week but it is too small.\nzh: 那我可以换一件大一点的吗？\nquit\n",
       keep=r"adapter |->|judge|Error|translated conversation|\[zh\]|\[en\]", label="6d translator works")
    print("\nPRE-FLIGHT PASSED: every step ran once with the real models. The long part starts now.")

In [ ]:
# 7. FINE-TUNING (LoRA on BMELD-train with context, in the empty-system-prompt chat format of the paper's fine-tuned model)
sh(f"{LAUNCH} scripts/finetune_lora.py --base_model {BASE_MODEL} --out_dir {ADAPTER_DIR} --time_budget_min {TRAIN_MINUTES}",
   keep=r"train examples|trainable|dev loss|\'loss\'|time budget|FINETUNE_DONE|Error", label="fine-tuning finished")

In [ ]:
# 8. Training summary + merge the adapter into the base weights (the standalone model used by the evaluation).
#    From here on a failing step is reported at the end instead of stopping the notebook: the adapter above is already saved.
import json, shutil
shutil.rmtree(f"{ADAPTER_DIR}/checkpoints", ignore_errors=True)
info = json.load(open(f"{ADAPTER_DIR}/train_info.json"))
print({k: info[k] for k in ("n_train_examples", "examples_seen", "steps", "train_seconds", "train_loss", "dev_loss_before", "dev_loss_after") if k in info})
sh(f"python scripts/merge_lora.py --adapter {ADAPTER_DIR} --out_dir {MERGED_DIR}", label="adapter merged", soft=True)

In [ ]:
# 9. Evaluation A: the base model, greedy, without / with context (the paper's baselines)
sh("bash scripts/run_zh_pipeline.sh", keep=r"^=+ |^###|Error|Traceback|FAILED", soft=True,
   env=dict(ROOT=RUN_ROOT, DATA_NAME="bmeld", SPLIT="test", MAX_DOCS=str(EVAL_DOCS), MAX_TOKENS="256", MODEL=BASE_MODEL,
            MODEL_NAME="TowerInstruct-7B-v0.2", STAGES="data instructions greedy eval"), label="base model evaluated")

In [ ]:
# 10. Evaluation B: the fine-tuned model = the paper's pipeline (greedy, sampled candidates, COMET MBR incl. context-aware, contrastive decoding, P-CXMI)
sh("bash scripts/run_zh_pipeline.sh", keep=r"^=+ |^###|Error|Traceback|FAILED", soft=True,
   env=dict(ROOT=RUN_ROOT, DATA_NAME="bmeld", SPLIT="test", MAX_DOCS=str(EVAL_DOCS), MAX_TOKENS="256", MODEL=MERGED_DIR, MODEL_NAME=FT_NAME,
            PROMPT_SUFFIX="_empty_sys", N_CANDIDATES=str(N_CANDIDATES), CONTEXT_SIZES="2", CD_VARIANTS="c1_nc1 c5_nc1",
            STAGES="greedy eval candidates mbr cd pcxmi report"), label="fine-tuned model evaluated")

In [ ]:
# 11. Consistency checks (prompt alignment, P-CXMI token gating, MBR selections, line counts): should end with "N/N checks passed"
sh(f"python scripts/check_zh_pipeline_outputs.py --root_dir {RUN_ROOT} --data_name bmeld --split test --model {MERGED_DIR} --model_name {FT_NAME} "
   f"--gen_backend vllm --n_candidates {N_CANDIDATES} --suffix _empty_sys --stages 'greedy eval candidates mbr cd pcxmi'",
   keep=r"FAIL|checks passed|sanity", label="checks done", soft=True)

In [ ]:
# 12. One results table with every system (like the paper's paper_results/*.csv) + COMET per message
sh(f"python scripts/consolidate_zh_results.py --root_dir {RUN_ROOT} --base_model_name TowerInstruct-7B-v0.2 --ft_model_name {FT_NAME} "
   f"--ft_suffix _empty_sys --gen_backend vllm --context_size 2 --comet_model {os.environ['COMET_MODEL']}", keep=r"^\||wrote|skipping", label="results table written", soft=True)

In [ ]:
# 13. Pack the small result files (candidates and model copies are left out), free the temporary copies and report
sh(f"cd {RUN_ROOT} && zip -qr /kaggle/working/results_zh.zip results_zh pipeline_report.json pcxmi_summary.json -x '*/candidates/*' && ls -lh /kaggle/working/results_zh.zip", soft=True)
for d in (MERGED_DIR, SMOKE_LORA, PREFLIGHT_ROOT):
    shutil.rmtree(d, ignore_errors=True)
print(f"TOTAL {(time.time() - T0) / 60:.0f} min. Kept as output: {ADAPTER_DIR} (adapter), {RUN_ROOT}, results_zh.zip")
print("ALL STEPS OK" if not FAILED else f"STEPS THAT FAILED (the adapter is saved anyway; read the log above): {FAILED}")

## Next
* **Save Version** (if this was an interactive run) so that the adapter `zh_lora/` is stored with this notebook.
* **Translate your own conversations:** open `translate_zh_kaggle.ipynb`, *Add Input → Notebook Output → this notebook*, run it.
* **Judge (optional):** `judge_zh_kaggle.ipynb` (CPU only) scores the outputs with Gemini and adds the scores to the results table.
* The results table is `run_zh/results_zh/bmeld/test.en-zh.csv` (with `test.en-zh.summary.md`). With 12 conversations the numbers are only a sanity check, not research results.